In [2]:
import os
from langgraph.graph import StateGraph
from langgraph.graph import END
from typing import TypedDict

In [3]:
from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_google_genai import ChatGoogleGenerativeAI

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [5]:
# Input schema for tools
class ToolInputSchema(TypedDict):
    query: str

In [6]:
# Shared State with structured input
class ToolState(TypedDict):
    input: ToolInputSchema
    output: str

In [7]:
# Tools Setup
wiki_tool = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
tavily_tool = TavilySearchResults()

In [8]:
# Wikipedia Node
def wikipedia_node(state: ToolState) -> ToolState:
    query = state["input"]["query"].strip()
    print(f"Wikipedia Searching for: {query}")
    
    result = wiki_tool.run(query)
    print("Wikipedia Result:", result)

    state["output"] = (
        result.strip() if isinstance(result, str) and result.strip()
        else "Wikipedia returned no result"
    )
    return state

In [9]:
# Tavily Node
def tavily_node(state: ToolState) -> ToolState:
    query = state["input"]["query"].strip()
    print(f"Tavily Searching for: {query}")
    
    result = tavily_tool.run(query)
    print("Raw Tavily Result:", result)

    if isinstance(result, list) and len(result) > 0:
        content = result[0].get("content", "").strip()
        state["output"] = content if content else "Tavily result has no content"
    elif isinstance(result, str) and result.strip():
        state["output"] = result.strip()
    else:
        state["output"] = "Tavily returned invalid or empty result"
    
    return state

In [10]:
# LangGraph Build
builder = StateGraph(ToolState)

In [11]:
builder.add_node("wiki_search", wikipedia_node)
builder.add_node("tavily_search", tavily_node)

In [12]:
# Flow: tavily ➝ fallback to wiki ➝ end
builder.set_entry_point("tavily_search")
builder.add_edge("tavily_search", "wiki_search")  # Fallback chaining
builder.add_edge("wiki_search", END)

In [13]:
graph = builder.compile()

In [15]:
# Test Run
state1 = {"input": {"query": "LangGraph AI"}, "output": ""}
result1 = graph.invoke(state1)

Tavily Searching for: LangGraph AI
Raw Tavily Result: [{'title': 'What is LangGraph? - IBM', 'url': 'https://www.ibm.com/think/topics/langgraph', 'content': 'LangGraph, created by LangChain, is an open source AI agent framework designed to build, deploy and manage complex generative AI agent workflows. It provides a set of tools and libraries that enable users to create, run and optimize large language models (LLMs) in a scalable and efficient manner. At its core, LangGraph uses the power of graph-based architectures to model and manage the intricate relationships between various components of an AI agent workflow. [...] Agent systems: LangGraph provides a framework for building agent-based systems, which can be used in applications such as robotics, autonomous vehicles or video games.\n\nLLM applications: By using LangGraph’s capabilities, developers can build more sophisticated AI models that learn and improve over time. Norwegian Cruise Line uses LangGraph to compile, construct and 

In [16]:
print("\nFinal Output:")
print("State:", result1)
print("Output:", result1.get("output", "'output' key missing")[:300])


Final Output:
State: {'input': {'query': 'LangGraph AI'}, 'output': 'Page: LangChain\nSummary: LangChain is a software framework that helps facilitate the integration of large language models (LLMs) into applications. As a language model integration framework, LangChain\'s use-cases largely overlap with those of language models in general, including document analysis and summarization, chatbots, and code analysis.\n\n\n\nPage: Applications of artificial intelligence\nSummary: Artificial intelligence (AI) has been used in applications throughout industry and academia. In a manner analogous to electricity or computers, AI serves as a general-purpose technology. AI programs are designed to simulate human perception and understanding. These systems are capable of adapting to new information and responding to changing situations. Machine learning has been used for various scientific and commercial purposes including language translation, image recognition, decision-making, credit scoring, 

In [17]:
# Another Run
state2 = {"input": {"query": "who is messi?"}, "output": ""}
result2 = graph.invoke(state2)

Tavily Searching for: who is messi?
Raw Tavily Result: [{'title': 'Lionel Messi - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/Lionel_Messi', 'content': 'Lionel Andrés "Leo" Messi( (Spanish pronunciation:( "Help:IPA/Spanish")ⓘ; born 24 June 1987) is an Argentine professional footballer who plays as a forward "Forward (association football)") for and captains "Captain (association football)") both Major League Soccer club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players in history, Messi set numerous records for individual accolades won throughout his professional footballing career such as eight Ballon [...] An Argentine international, Messi is the national team\'s all-time leading goalscorer and most-capped player. His style of play as a diminutive, left-footed dribbler, drew career-long comparisons with compatriot Diego Maradona, who described Messi as his successor. At the youth level, he won the 2005 FIFA World Youth Championship and 

In [18]:
print("\nFinal Output:")
print("State:", result2)
print("Output:", result2.get("output", "'output' key missing")[:300])


Final Output:
State: {'input': {'query': 'who is messi?'}, 'output': 'Page: Lionel Messi\nSummary: Lionel Andrés "Leo" Messi (Spanish pronunciation: [ljoˈnel anˈdɾes ˈmesi] ; born 24 June 1987) is an Argentine professional footballer who plays as a forward for and captains both Major League Soccer club Inter Miami and the Argentina national team. Widely regarded as one of the greatest players in history, Messi set numerous records for individual accolades won throughout his professional footballing career, including eight Ballon d\'Ors, six European Golden Shoes, and eight times being named the world\'s best player by FIFA. He is the most decorated player in the history of professional football having won 45 team trophies. Messi\'s records include most goals in a calendar year (91), most goal contributions (21) in the FIFA World Cup, and most goal contributions (32) in the Copa América. A prolific goalscorer and creative playmaker, Messi has scored over 850 senior career goals and has